# Feature Engineering

## Import modules

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import scipy.sparse as sp
import joblib
from sentence_transformers import SentenceTransformer
import numpy as np

## Import data

In [2]:
df = pd.read_parquet('csv/preprocess.parquet')
df.head()

,Sentiment,Text Content,tweet_length,Mutated Text Content,Text Tokens
0,1,"Thanks, @MSFTImagine & @IamPablo kindly sendin...",104,"thanks, @msftimagine & @iampablo kindly sendin...","[thanks, kindly, sending, awesome, microsoft, ..."
1,1,Just bought GTA I’m done,24,just bought gta i am done,"[bought, gta, done]"
2,1,UGH SO AWESOME!!!,17,ugh so awesome!!!,"[ugh, awesome]"
3,1,It's no secret that I love Assassin's Creed. I...,175,it is no secret that i love assassin's creed. ...,"[no, secret, love, assassins, creed, extremely..."
4,1,WORLD LIVE IN TELUGU | BE MUCH FUN | BE LIVE,44,world live in telugu | be much fun | be live,"[world, live, telugu, much, fun, live]"


# TF-IDF

TF-IDF (term frequency - inverse document frequency) is used in natural language processing models (like the one I am making) to calculate the importance of a word in a document/block of text by using statistics. It is used to extract the features (like the value of a word compared to its sentiment). 

## TF
TF (term frequency) measures how often a word shows up in a document. A higher frequency suggests greater importance. If it shows up frequently, it likely means it is highly relevant to the document. 
<!-- For anyone viewing the source of this, I used [GFG](https://www.geeksforgeeks.org/machine-learning/understanding-tf-idf-term-frequency-inverse-document-frequency/) for the definitions -->

## IDF
IDF is the inverse document frequency and reduces the weight of common words within the document while increasing the weight of rare words. If a term (such as "this") appears more likely than ("newtonian"), it is less important, while "newtonian" is more important and its weighting gets raised.  

## Calculating
Calculating is simple, as it is simply the product of the two:
$$
\text{TF-IDF(t, d, D)} = \text{TF(t, d)} \times \text{IDF(t, D)}
$$

## Implementation

We use a type of TF-IDF feature extractor called TfidfVectoriser, which just takes a word from our tokenised values (also done in preprocessing) and passes it into the TF-IDF function, which returns a weighting for the word. 

This is then stored as a feature matrix (rows = tweets/documents, columns = vocabulary, values = TF-IDF weights). This can then be used as input for Logistic Regression (which is done in model training).

In [3]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2) # allows for such features like "not good"
)
X_tfidf = tfidf.fit_transform(df['Mutated Text Content'])

X = sp.hstack([X_tfidf])
y = df['Sentiment']

Within the vectoriser, I have used a ngram_range as an argument. This combines tokens if the context is relevant, and allows for features like ["not good"] instead of ["not", "good"], which might lose context with the negator. 

Within this data, we have got 5000 features (as specified by the TfidfVectorizer) and have fitted the imported CSV `Text Tokens` column as the fit. 

The Y is the `Sentiment` column.

# Issue

TF-IDF uses a bag-of-words approach, creating a correlation between the frequency of words and their importance score. It's great at fetching the sentiment of individual words, however because it treats each word in isolation, necessary context is lost.

Take an example:

`This is fucking awesome` (excuse my profranity, but it is a good example)

You (the reader) would perceive this word to be as a generally positive sentiment. The model would tokenise each word into `This`, `is`, `fucking`, `awesome`. Based on its training data, it would consider `fucking` to be highly negative and `awesome` to be highly positive, so it gets confused, eventually leading to a score of **negative** with an overall confidence of 50.68%.

# SBERT

SBERT (or Sentence Bidirectional Encoder Representations from Transformers) is a model used to embed the context of a sentence into tokens of a word.

It takes an entire sequence of words and squashes it down to a list of 384 numbers/dimensions that capture its meaning.
During training, sentences that mean similar things end up with a similar list of numbers, while opposite things end up with a different set of numbers and are placed in a different dimension.

The 'B' in BERT is used to signify how the model uses bidirection to gather context to see how the neighboring words affect each other.

## Implementation

In [4]:
sbert = SentenceTransformer("all-MiniLM-L6-v2")
texts = df["Mutated Text Content"].fillna("").tolist()

x_sbert = sbert.encode(texts, convert_to_numpy=True, show_progress_bar=True)

X = sp.csr_matrix(x_sbert[:, :128].astype(np.float32))
y = df["Sentiment"]

print("SBERT feature shape:", X.shape)
print("Label shape:", y.shape)

Batches:   0%|          | 0/16869 [00:00<?, ?it/s]

SBERT feature shape: (539794, 128)
Label shape: (539794,)


# Exporting

I think it is ready to export the vectoriser for training to use. 

In [5]:
sp.save_npz("csv/x_features.npz", X)
y.to_csv("csv/y_sentiment_labels.csv", index=False)